#  Week 7 Assignment: Retrieval-Augmented Generation (RAG) Based Document Question Answering System

## Assignment Description

This project aims to develop a simple **Retrieval-Augmented Generation (RAG)** system capable of answering user questions from custom documents such as PDF files, text files, or domain-specific datasets. The system retrieves the most relevant information from the provided documents using vector similarity search and generates accurate, context-aware responses with the help of a Large Language Model (LLM).

The implementation covers the complete RAG pipeline, including document ingestion, text chunking, embedding generation, vector database creation, semantic retrieval, and answer generation. The project also evaluates retrieval performance using validation examples and reports important system metrics such as chunking configuration, embedding model, vector database, and language model setup.

## Objectives

- Build an end-to-end Retrieval-Augmented Generation (RAG) pipeline.
- Load and process custom documents for question answering.
- Generate semantic embeddings and store them in a vector database.
- Retrieve relevant document chunks using similarity search.
- Generate grounded and context-aware responses using an LLM.
- Evaluate the system through validation examples and performance metrics.

# Step 1: Install Required Libraries



This step installs all the libraries required to build the Retrieval-Augmented Generation (RAG) system. These libraries provide functionality for document loading, text preprocessing, embedding generation, vector database management, retrieval, and interaction with the Large Language Model. Installing these dependencies prepares the environment for implementing the complete question-answering pipeline.

In [26]:


!pip -q install -U \
langchain \
langchain-community \
langchain-text-splitters \
langchain-google-genai \
langchain-huggingface \
faiss-cpu \
sentence-transformers \
pypdf \
python-dotenv

print("✅ All required libraries installed successfully.")

✅ All required libraries installed successfully.


# Step 2: Import Required Libraries


In this step, we import all the required modules that will be used throughout the project. These libraries provide functionalities for document loading, text chunking, embedding generation, vector database creation, retrieval, and interaction with the Large Language Model. Importing them at the beginning of the notebook keeps the workflow organized and ensures all required components are readily available for the subsequent steps.

In [27]:


# Standard Libraries
import os
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# Environment Variables
from dotenv import load_dotenv

# Document Loader
from langchain_community.document_loaders import PyPDFLoader

# Text Splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding Model
from langchain_huggingface import HuggingFaceEmbeddings

# Vector Database
from langchain_community.vectorstores import FAISS

# Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI

# Prompt Template
from langchain_core.prompts import PromptTemplate

# Load environment variables
load_dotenv()

print("✅ All required modules imported successfully!")

✅ All required modules imported successfully!


# Step 3: Load the Custom Document



In this step, we load the custom document that will serve as the knowledge source for the Retrieval-Augmented Generation (RAG) system. The document is read using the **PyPDFLoader**, which extracts the textual content from each page of the PDF and converts it into a format suitable for further processing. Successfully loading the document is the first step in enabling semantic search and question answering over its contents.

In [28]:
from google.colab import files

# Open file picker
uploaded = files.upload()

# Get the uploaded file name
pdf_path = list(uploaded.keys())[0]

# Load the PDF
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Display document information
print("=" * 60)
print("📄 Document Loaded Successfully")
print("=" * 60)
print(f"Document Name : {pdf_path}")
print(f"Total Pages   : {len(documents)}")
print("=" * 60)

# Preview the first page
print("\n📖 Preview of First Page:\n")
print(documents[0].page_content[:1000])

Saving Sushant_Kurund_Resume.pdf to Sushant_Kurund_Resume (1).pdf
📄 Document Loaded Successfully
Document Name : Sushant_Kurund_Resume (1).pdf
Total Pages   : 2

📖 Preview of First Page:

Sushant Kurund 
 
Roshan Milestone E-401, Tathawade, Pune – 411033 
Email: sushantdkurund225@gmail.com | Mobile: 7276758125 
 
Objective 
Motivated and detail-oriented MCA student with a strong foundation in web development 
and a growing interest in Data Science. Skilled in Python, Java, and front-end technologies, 
with hands-on project experience. 
Technical Skills 
Programming Languages: Java, Python, JavaScript 
Web Technologies: HTML, CSS, JavaScript 
Frameworks & Tools: Django (Basic), Git 
Database: SQL (Basic), DBMS concepts 
Projects 
Laundry Service Application (UG Project) 
- Developed using Java, JavaScript, HTML, CSS, and Servlet. 
- Features: user registration, service selection, order management. 
E-Verify System (MCA Project) 
- Built using Django, Python, JavaScript, HTML, CSS. 
- Im

# Step 4: Split the Document into Chunks
In this step, the extracted document text is divided into smaller overlapping chunks using the **RecursiveCharacterTextSplitter**. Chunking improves retrieval performance by breaking large documents into manageable pieces while preserving context through overlapping text. These chunks will later be converted into vector embeddings for semantic similarity search.

In [29]:
# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

# Split the document into chunks
chunks = text_splitter.split_documents(documents)

# Display chunk statistics
print("=" * 60)
print("📄 Document Chunking Completed")
print("=" * 60)
print(f"Chunk Size      : 500 characters")
print(f"Chunk Overlap   : 50 characters")
print(f"Total Chunks    : {len(chunks)}")
print("=" * 60)

# Display previews of the first three chunks
preview_count = min(3, len(chunks))

for i in range(preview_count):
    print(f"\n📌 Chunk {i+1}")
    print("-" * 60)
    print(chunks[i].page_content[:500])

📄 Document Chunking Completed
Chunk Size      : 500 characters
Chunk Overlap   : 50 characters
Total Chunks    : 4

📌 Chunk 1
------------------------------------------------------------
Sushant Kurund 
 
Roshan Milestone E-401, Tathawade, Pune – 411033 
Email: sushantdkurund225@gmail.com | Mobile: 7276758125 
 
Objective 
Motivated and detail-oriented MCA student with a strong foundation in web development 
and a growing interest in Data Science. Skilled in Python, Java, and front-end technologies, 
with hands-on project experience. 
Technical Skills 
Programming Languages: Java, Python, JavaScript 
Web Technologies: HTML, CSS, JavaScript

📌 Chunk 2
------------------------------------------------------------
Web Technologies: HTML, CSS, JavaScript 
Frameworks & Tools: Django (Basic), Git 
Database: SQL (Basic), DBMS concepts 
Projects 
Laundry Service Application (UG Project) 
- Developed using Java, JavaScript, HTML, CSS, and Servlet. 
- Features: user registration, service selectio

# Step 5: Generate Text Embeddings



In this step, each text chunk is converted into a numerical vector representation called an **embedding** using a pre-trained Sentence Transformer model. These embeddings capture the semantic meaning of the text, allowing the system to compare documents based on their meaning rather than exact keywords. The generated embeddings will later be stored in a FAISS vector database to enable efficient similarity-based retrieval.

In [30]:
# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("=" * 60)
print("🧠 Embedding Model Initialized")
print("=" * 60)
print(f"Model Name          : sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding Dimension : 384")
print("=" * 60)

# Generate an embedding for the first chunk (for verification)
sample_embedding = embedding_model.embed_query(chunks[0].page_content)

print("\n✅ Embedding generated successfully!")
print(f"Vector Length : {len(sample_embedding)}")

# Display the first 10 values of the embedding vector
print("\nFirst 10 Embedding Values:")
print(sample_embedding[:10])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🧠 Embedding Model Initialized
Model Name          : sentence-transformers/all-MiniLM-L6-v2
Embedding Dimension : 384

✅ Embedding generated successfully!
Vector Length : 384

First 10 Embedding Values:
[-0.07079331576824188, 0.007411560975015163, -0.03998076915740967, -0.010945316404104233, 0.00704933563247323, -0.13102847337722778, -0.00220455857925117, 0.04078741371631622, -0.1085418090224266, -0.014911233447492123]


# Step 6: Create the FAISS Vector Database


In this step, the generated text embeddings are stored in a **FAISS (Facebook AI Similarity Search)** vector database. FAISS enables efficient semantic similarity search by indexing the embedding vectors, allowing the system to quickly retrieve the most relevant document chunks for a given user query. This forms the retrieval component of the Retrieval-Augmented Generation (RAG) pipeline.

In [31]:
# Create the FAISS vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("=" * 60)
print("📚 FAISS Vector Database Created Successfully")
print("=" * 60)
print(f"Total Document Chunks Indexed : {len(chunks)}")
print(f"Embedding Model               : sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding Dimension           : 384")
print(f"Vector Database              : FAISS")
print("=" * 60)

📚 FAISS Vector Database Created Successfully
Total Document Chunks Indexed : 4
Embedding Model               : sentence-transformers/all-MiniLM-L6-v2
Embedding Dimension           : 384
Vector Database              : FAISS


# Step 7: Configure the Document Retriever



In this step, a retriever is created from the FAISS vector database to perform semantic similarity searches over the indexed document chunks. When a user submits a question, the retriever identifies and returns the most relevant chunks based on vector similarity. These retrieved chunks will later be provided to the language model as contextual information for generating accurate and grounded responses.

In [32]:
# Create a retriever from the FAISS vector database
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

print("=" * 60)
print("🔍 Document Retriever Configured Successfully")
print("=" * 60)
print("Search Type          : Similarity Search")
print("Top Chunks Retrieved : 3")
print("Vector Store         : FAISS")
print("=" * 60)

🔍 Document Retriever Configured Successfully
Search Type          : Similarity Search
Top Chunks Retrieved : 3
Vector Store         : FAISS


# Step 8: Configure the Google Gemini Language Model



In this step, the Google Gemini language model is initialized using an API key. The language model receives the user's question along with the retrieved document context and generates accurate, context-aware responses. Setting the temperature to **0.3** helps produce more consistent and deterministic answers while minimizing randomness.

In [33]:
import os
from getpass import getpass

# Enter your Gemini API Key
os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")

# Initialize Gemini
llm = ChatGoogleGenerativeAI(
   model="gemini-flash-latest",
    temperature=0.3
)

print("=" * 60)
print("🤖 Google Gemini Initialized Successfully")
print("=" * 60)
print("Model        : gemini-2.5-flash")
print("Temperature  : 0.3")
print("=" * 60)

Enter your Gemini API Key: ··········
🤖 Google Gemini Initialized Successfully
Model        : gemini-2.5-flash
Temperature  : 0.3


# Step 9: Create the Retrieval-Augmented Generation (RAG) Pipeline


In this step, the Retrieval-Augmented Generation (RAG) pipeline is constructed by combining the document retriever with the Google Gemini language model. A custom prompt template is created to provide the retrieved document context along with the user's question to the language model. This ensures that all generated responses are grounded in the retrieved document content rather than relying solely on the model's internal knowledge.

In [34]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt Template
prompt = PromptTemplate(
    template="""
You are a helpful AI assistant.

Answer the question ONLY using the context provided below.

If the answer is not present in the context, reply:
"I couldn't find the answer in the provided document."

Context:
{context}

Question:
{question}

Answer:
""",
    input_variables=["context", "question"]
)

# Function to format retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG Pipeline
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("=" * 60)
print("✅ RAG Pipeline Created Successfully")
print("=" * 60)
print("Retriever      : FAISS Similarity Search")
print("Language Model : Gemini 2.5 Flash")
print("Pipeline Status: Ready for Question Answering")
print("=" * 60)

✅ RAG Pipeline Created Successfully
Retriever      : FAISS Similarity Search
Language Model : Gemini 2.5 Flash
Pipeline Status: Ready for Question Answering


# Step 10: Query the RAG System



In this step, the completed Retrieval-Augmented Generation (RAG) pipeline is used to answer user questions. The user's query is passed through the retriever, which identifies the most relevant document chunks from the FAISS vector database. These retrieved chunks are then supplied to the Google Gemini language model, enabling it to generate accurate, context-aware responses based solely on the document content.

In [35]:
# Enter your question
question = "What programming languages does the candidate know?"

print("=" * 60)
print("❓ User Question")
print("=" * 60)
print(question)

print("\nGenerating answer...\n")

# Generate answer
answer = rag_chain.invoke(question)

print("=" * 60)
print("🤖 Generated Answer")
print("=" * 60)
print(answer)

❓ User Question
What programming languages does the candidate know?

Generating answer...

🤖 Generated Answer
Based on the provided document, the candidate knows the following programming languages:
- Java
- Python
- JavaScript


# Step 11: Validate the RAG System



This step evaluates the Retrieval-Augmented Generation (RAG) system using multiple sample questions. For each query, the system retrieves the most relevant document chunks from the FAISS vector database and then generates a context-aware response using the Google Gemini language model. Displaying both the retrieved context and the generated answer verifies that the responses are grounded in the uploaded document rather than relying solely on the language model's internal knowledge.

In [37]:
import time

# Sample questions for validation
sample_questions = [
    "What programming languages does the candidate know?",
    "What projects has the candidate completed?",
    "What is the candidate's MCA specialization?",
    "What is the candidate's UG CGPA?",
    "Which frameworks and tools does the candidate know?",
    "What is the candidate's email address?"
]

print("=" * 100)
print("🧪 RAG SYSTEM VALIDATION")
print("=" * 100)

for i, question in enumerate(sample_questions, start=1):

    print(f"\n🔹 Question {i}")
    print("-" * 100)
    print(f"Question: {question}")

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    print("\n📄 Retrieved Context:")
    print("-" * 100)

    for j, doc in enumerate(retrieved_docs, start=1):
        print(f"\nContext {j}:")
        print(doc.page_content)

    # Generate answer with automatic retry
    while True:
        try:
            answer = rag_chain.invoke(question)
            break
        except Exception as e:
            print("\n⚠️ Rate limit reached.")
            print("Waiting 6 seconds before retrying...")
            time.sleep(6)

    print("\n🤖 Generated Answer:")
    print("-" * 100)
    print(answer)

    print("\n" + "=" * 100)

    # Wait before sending the next request
    time.sleep(6)

🧪 RAG SYSTEM VALIDATION

🔹 Question 1
----------------------------------------------------------------------------------------------------
Question: What programming languages does the candidate know?

📄 Retrieved Context:
----------------------------------------------------------------------------------------------------

Context 1:
Web Technologies: HTML, CSS, JavaScript 
Frameworks & Tools: Django (Basic), Git 
Database: SQL (Basic), DBMS concepts 
Projects 
Laundry Service Application (UG Project) 
- Developed using Java, JavaScript, HTML, CSS, and Servlet. 
- Features: user registration, service selection, order management. 
E-Verify System (MCA Project) 
- Built using Django, Python, JavaScript, HTML, CSS. 
- Implemented authentication and verification workflows. 
Education 
MCA (Pursuing) – Focus: Data Science

Context 2:
Sushant Kurund 
 
Roshan Milestone E-401, Tathawade, Pune – 411033 
Email: sushantdkurund225@gmail.com | Mobile: 7276758125 
 
Objective 
Motivated and detail-

# Step 12: Configuration Summary

The following table summarizes the key configuration choices used while implementing the Retrieval-Augmented Generation (RAG) pipeline.

In [38]:
from pandas import DataFrame

config = {
    "Component": [
        "Document",
        "Chunk Size",
        "Chunk Overlap",
        "Embedding Model",
        "Embedding Dimension",
        "Vector Database",
        "Retriever",
        "Top-K Retrieval",
        "Language Model"
    ],
    "Configuration": [
        pdf_path,
        500,
        50,
        "sentence-transformers/all-MiniLM-L6-v2",
        384,
        "FAISS",
        "Similarity Search",
        3,
        "gemini-flash-latest"
    ]
}

DataFrame(config)

,Component,Configuration
0,Document,Sushant_Kurund_Resume (1).pdf
1,Chunk Size,500
2,Chunk Overlap,50
3,Embedding Model,sentence-transformers/all-MiniLM-L6-v2
4,Embedding Dimension,384
5,Vector Database,FAISS
6,Retriever,Similarity Search
7,Top-K Retrieval,3
8,Language Model,gemini-flash-latest


# Conclusion

## Summary

An end-to-end Retrieval-Augmented Generation (RAG) system was successfully implemented using LangChain, FAISS, Hugging Face sentence-transformer embeddings, and Google Gemini.

The pipeline performs the following tasks:

- Loads PDF documents.
- Splits text into overlapping chunks.
- Generates dense vector embeddings.
- Stores embeddings inside a FAISS vector database.
- Retrieves the most relevant document chunks.
- Generates grounded responses using Gemini.

Validation using multiple sample queries demonstrated that the system retrieves relevant context before generating responses, reducing hallucinations and improving answer accuracy.

This implementation satisfies all required objectives of the assignment and demonstrates a complete Retrieval-Augmented Generation workflow.